# Trial 3 — Hardware-Grounded Phase

**Phase 3 of the Adaptive CNN Acceleration study.** Trial 1 used an analytical
roofline model; Trial 2 used a functional emulator with *hand-set* FPGA
efficiency constants. This notebook replaces those hand-set FPGA op efficiencies
with values **synthesized by Vitis HLS for the actual Zynq-7020 (xc7z020)** — see
`hls/` and `hls/reports/_fpga_constants.json`.

The pipeline (layer model, builders, fusion, GPU emulator, partition optimizer)
is **identical to Trial 2** so the only variable is the FPGA cost model. This is
a controlled ablation: *what changes when the FPGA constants are grounded in
real synthesis instead of guessed?*

`trial1.ipynb` and `trial2.ipynb` are left untouched.


In [ ]:
# Cell 1 - Imports, setup, and load HLS-synthesized FPGA constants
import os, json
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple
from enum import Enum

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

OUTPUT_DIR = "ResultImages_groundedPhase"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Transfer model (same as trial2): PCIe Gen3 x4
TARGET_XFER_BW_GBPS = 4.0
TARGET_XFER_SETUP_US = 50.0

FPGA_DSP_SLICES = 220   # xc7z020

# ----- Load the HLS-synthesized per-op efficiency (MACs/cycle/DSP) -----
HLS_JSON = os.path.join("hls", "reports", "_fpga_constants.json")
with open(HLS_JSON) as f:
    hls = json.load(f)

print("HLS-synthesized FPGA constants (xc7z020):")
for op, d in hls.items():
    print(f"  {op:16s} eff/DSP={d['efficiency_per_dsp']:.3f}  "
          f"MACs/cyc={d['macs_per_cycle']:.2f}  DSP={d['dsp']}")

## Mapping HLS results to the emulator's FPGA efficiency

Trial 2's `FPGAEmulator` models FPGA compute cost as
`mac_ops / (DSP_SLICES * efficiency)`, where `efficiency ∈ (0,1]` is a per-op
utilization factor — hand-set in Trial 2. We replace it with the **synthesized
`eff/DSP`** (MACs per cycle per DSP) from `hls/reports/`, i.e. how close each
op's real datapath gets to one MAC per DSP per cycle on the xc7z020.

`CONV`, `DEPTHWISE_CONV`, `POINTWISE_CONV`, `BATCHNORM` come directly from
synthesis. `RELU`, `POOL`, `FC` were **not** synthesized and are assigned by
analogy (elementwise ops ≈ batchnorm; FC ≈ pointwise matmul) — flagged below.

**Caveats (see `hls/README.md`):**
1. The synthesized kernels have *unmatched parallelism* (conv is unrolled,
   others single-lane), so `eff/DSP` is the fair normalized metric but is a
   first-pass estimate.
2. INT8 is already baked into the HLS kernels; we keep Trial 2's `emulate_int8`
   step for a like-for-like ablation, which slightly double-counts — it applies
   equally to grounded and hand-set models, so it cancels in the comparison.


In [ ]:
# Cell 2 (from trial2) - Layer definitions and architecture builders
class LayerType(Enum):
    CONV = "conv"
    DEPTHWISE_CONV = "depthwise_conv"
    POINTWISE_CONV = "pointwise_conv"
    BATCHNORM = "batchnorm"
    RELU = "relu"
    POOL = "pool"
    FC = "fc"

@dataclass
class Layer:
    name: str
    layer_type: LayerType
    in_channels: int
    in_height: int
    in_width: int
    out_channels: int
    out_height: int
    out_width: int
    kernel_size: int = 1
    stride: int = 1
    groups: int = 1

    @property
    def mac_ops(self) -> int:
        if self.layer_type == LayerType.CONV:
            return (self.kernel_size ** 2) * self.in_channels * self.out_channels * self.out_height * self.out_width
        elif self.layer_type == LayerType.DEPTHWISE_CONV:
            return (self.kernel_size ** 2) * self.in_channels * self.out_height * self.out_width
        elif self.layer_type == LayerType.POINTWISE_CONV:
            return self.in_channels * self.out_channels * self.out_height * self.out_width
        elif self.layer_type == LayerType.BATCHNORM:
            return 2 * self.out_channels * self.out_height * self.out_width
        elif self.layer_type == LayerType.RELU:
            return self.out_channels * self.out_height * self.out_width
        elif self.layer_type == LayerType.POOL:
            return (self.kernel_size ** 2) * self.out_channels * self.out_height * self.out_width
        elif self.layer_type == LayerType.FC:
            return self.in_channels * self.out_channels
        return 0

    @property
    def param_count(self) -> int:
        if self.layer_type == LayerType.CONV:
            return self.kernel_size ** 2 * self.in_channels * self.out_channels
        elif self.layer_type == LayerType.DEPTHWISE_CONV:
            return self.kernel_size ** 2 * self.in_channels
        elif self.layer_type == LayerType.POINTWISE_CONV:
            return self.in_channels * self.out_channels
        elif self.layer_type == LayerType.BATCHNORM:
            return 2 * self.out_channels
        elif self.layer_type == LayerType.FC:
            return self.in_channels * self.out_channels
        return 0

    @property
    def output_tensor_size_bytes(self) -> int:
        return 4 * self.out_channels * self.out_height * self.out_width

    @property
    def input_tensor_size_bytes(self) -> int:
        return 4 * self.in_channels * self.in_height * self.in_width

    @property
    def memory_bytes_accessed(self) -> int:
        return self.input_tensor_size_bytes + self.output_tensor_size_bytes + self.param_count * 4


def build_resnet18() -> List[Layer]:
    layers = []
    layers.append(Layer("conv1", LayerType.CONV, 3, 224, 224, 64, 112, 112, kernel_size=7, stride=2))
    layers.append(Layer("bn1", LayerType.BATCHNORM, 64, 112, 112, 64, 112, 112))
    layers.append(Layer("relu1", LayerType.RELU, 64, 112, 112, 64, 112, 112))
    layers.append(Layer("maxpool", LayerType.POOL, 64, 112, 112, 64, 56, 56, kernel_size=3, stride=2))
    for g, (c, h) in enumerate([(64, 56), (128, 28), (256, 14), (512, 7)], 1):
        for b in range(2):
            in_c = (c // 2) if (b == 0 and g > 1) else c
            h_in = (h * 2) if (b == 0 and g > 1) else h
            s = 2 if (b == 0 and g > 1) else 1
            prefix = f"layer{g}_b{b}"
            layers.append(Layer(f"{prefix}_conv1", LayerType.CONV, in_c, h_in, h_in, c, h, h, kernel_size=3, stride=s))
            layers.append(Layer(f"{prefix}_bn1", LayerType.BATCHNORM, c, h, h, c, h, h))
            layers.append(Layer(f"{prefix}_relu1", LayerType.RELU, c, h, h, c, h, h))
            layers.append(Layer(f"{prefix}_conv2", LayerType.CONV, c, h, h, c, h, h, kernel_size=3))
            layers.append(Layer(f"{prefix}_bn2", LayerType.BATCHNORM, c, h, h, c, h, h))
            layers.append(Layer(f"{prefix}_relu2", LayerType.RELU, c, h, h, c, h, h))
    layers.append(Layer("avgpool", LayerType.POOL, 512, 7, 7, 512, 1, 1, kernel_size=7))
    layers.append(Layer("fc", LayerType.FC, 512, 1, 1, 1000, 1, 1))
    return layers


def build_mobilenetv2() -> List[Layer]:
    layers = []
    layers.append(Layer("conv_stem", LayerType.CONV, 3, 224, 224, 32, 112, 112, kernel_size=3, stride=2))
    layers.append(Layer("bn_stem", LayerType.BATCHNORM, 32, 112, 112, 32, 112, 112))
    layers.append(Layer("relu_stem", LayerType.RELU, 32, 112, 112, 32, 112, 112))
    block_configs = [
        (32, 32, 16, 112, 56, 2), (16, 96, 24, 56, 28, 2),
        (24, 144, 24, 28, 28, 1), (24, 144, 32, 28, 14, 2),
        (32, 192, 32, 14, 14, 1), (32, 192, 32, 14, 14, 1),
        (32, 192, 64, 14, 7, 2), (64, 384, 64, 7, 7, 1),
        (64, 384, 64, 7, 7, 1), (64, 384, 96, 7, 7, 1),
        (96, 576, 96, 7, 7, 1), (96, 576, 96, 7, 7, 1),
        (96, 576, 160, 7, 4, 2), (160, 960, 160, 4, 4, 1),
        (160, 960, 160, 4, 4, 1), (160, 960, 320, 4, 4, 1),
    ]
    for idx, (in_c, exp_c, out_c, h_in, h_out, stride) in enumerate(block_configs):
        prefix = f"block{idx+1}"
        if exp_c != in_c:
            layers.append(Layer(f"{prefix}_expand", LayerType.POINTWISE_CONV, in_c, h_in, h_in, exp_c, h_in, h_in))
            layers.append(Layer(f"{prefix}_expand_bn", LayerType.BATCHNORM, exp_c, h_in, h_in, exp_c, h_in, h_in))
            layers.append(Layer(f"{prefix}_expand_relu", LayerType.RELU, exp_c, h_in, h_in, exp_c, h_in, h_in))
        layers.append(Layer(f"{prefix}_dw", LayerType.DEPTHWISE_CONV, exp_c, h_in, h_in, exp_c, h_out, h_out, kernel_size=3, stride=stride))
        layers.append(Layer(f"{prefix}_dw_bn", LayerType.BATCHNORM, exp_c, h_out, h_out, exp_c, h_out, h_out))
        layers.append(Layer(f"{prefix}_dw_relu", LayerType.RELU, exp_c, h_out, h_out, exp_c, h_out, h_out))
        layers.append(Layer(f"{prefix}_project", LayerType.POINTWISE_CONV, exp_c, h_out, h_out, out_c, h_out, h_out))
        layers.append(Layer(f"{prefix}_project_bn", LayerType.BATCHNORM, out_c, h_out, h_out, out_c, h_out, h_out))
    layers.append(Layer("conv_final", LayerType.POINTWISE_CONV, 320, 4, 4, 1280, 4, 4))
    layers.append(Layer("bn_final", LayerType.BATCHNORM, 1280, 4, 4, 1280, 4, 4))
    layers.append(Layer("relu_final", LayerType.RELU, 1280, 4, 4, 1280, 4, 4))
    layers.append(Layer("avgpool", LayerType.POOL, 1280, 4, 4, 1280, 1, 1, kernel_size=4))
    layers.append(Layer("fc", LayerType.FC, 1280, 1, 1, 1000, 1, 1))
    return layers

print("Architectures:",
      len(build_resnet18()), "ResNet-18 layers,",
      len(build_mobilenetv2()), "MobileNetV2 layers")

In [ ]:
# Cell 3 (from trial2) - Operator fusion (structural; no torch verification here)
FUSIBLE_PATTERNS = [
    ("conv_bn_relu",   [LayerType.CONV, LayerType.BATCHNORM, LayerType.RELU]),
    ("dw_bn_relu",     [LayerType.DEPTHWISE_CONV, LayerType.BATCHNORM, LayerType.RELU]),
    ("pw_bn_relu",     [LayerType.POINTWISE_CONV, LayerType.BATCHNORM, LayerType.RELU]),
    ("conv_bn",        [LayerType.CONV, LayerType.BATCHNORM]),
    ("dw_bn",          [LayerType.DEPTHWISE_CONV, LayerType.BATCHNORM]),
    ("pw_bn",          [LayerType.POINTWISE_CONV, LayerType.BATCHNORM]),
]

@dataclass
class FusedBlock:
    name: str
    pattern: str
    layers: List[Layer]
    @property
    def total_mac_ops(self): return sum(l.mac_ops for l in self.layers)
    @property
    def total_params(self): return sum(l.param_count for l in self.layers)
    @property
    def input_tensor_size_bytes(self): return self.layers[0].input_tensor_size_bytes
    @property
    def output_tensor_size_bytes(self): return self.layers[-1].output_tensor_size_bytes
    @property
    def dominant_layer_type(self): return self.layers[0].layer_type

def apply_fusion(layers: List[Layer]) -> List:
    sorted_patterns = sorted(FUSIBLE_PATTERNS, key=lambda p: len(p[1]), reverse=True)
    result, i, fused_count = [], 0, 0
    while i < len(layers):
        matched = False
        for pattern_name, pattern_types in sorted_patterns:
            pat_len = len(pattern_types)
            if i + pat_len <= len(layers):
                cand = [layers[i + j] for j in range(pat_len)]
                if all(c.layer_type == pt for c, pt in zip(cand, pattern_types)):
                    fused_count += 1
                    result.append(FusedBlock(f"fused_{fused_count}_{pattern_name}", pattern_name, cand))
                    i += pat_len; matched = True; break
        if not matched:
            result.append(layers[i]); i += 1
    return result

for name, b in [("ResNet-18", build_resnet18), ("MobileNetV2", build_mobilenetv2)]:
    fb = apply_fusion(b())
    print(f"{name}: {len(b())} layers -> {len(fb)} fused blocks")

In [ ]:
# Cell 4 (from trial2) - GPU + FPGA emulators (identical model; FPGA eff dict swapped)
@dataclass
class EmuResult:
    name: str; mac_ops: int; memory_bytes: int
    compute_units: float; memory_units: float; effective_units: float
    is_fused: bool = False; fusion_memory_saved: int = 0

class FPGAEmulator:
    def __init__(self, dsp_slices=FPGA_DSP_SLICES, mem_bandwidth_ratio=1.0, streaming_efficiency=None):
        self.dsp_slices = dsp_slices
        self.mem_bandwidth_ratio = mem_bandwidth_ratio
        self.streaming_efficiency = streaming_efficiency
    def emulate_layer(self, layer):
        eff = self.streaming_efficiency[layer.layer_type]
        compute_units = layer.mac_ops / (self.dsp_slices * eff)
        effective_bw = self.mem_bandwidth_ratio * eff
        memory_units = layer.memory_bytes_accessed / max(effective_bw, 0.01)
        eu = max(compute_units, memory_units)
        return EmuResult(layer.name, layer.mac_ops, layer.memory_bytes_accessed, compute_units, memory_units, eu)
    def emulate_fused_block(self, layers, pattern_name=""):
        tmac=tmem=tcu=fsaved=0
        for i, layer in enumerate(layers):
            r = self.emulate_layer(layer); tmac += layer.mac_ops; tcu += r.compute_units
            if i == 0:
                tmem += layer.input_tensor_size_bytes + layer.param_count*4
            elif i == len(layers)-1:
                tmem += layer.param_count*4 + layer.output_tensor_size_bytes
            else:
                tmem += layer.param_count*4
            if i < len(layers)-1: fsaved += layer.output_tensor_size_bytes*2
        avg_eff = np.mean([self.streaming_efficiency[l.layer_type] for l in layers])
        mu = tmem / max(self.mem_bandwidth_ratio*avg_eff, 0.01)
        return EmuResult(f"fused_{pattern_name}", tmac, tmem, tcu, mu, max(tcu, mu), True, fsaved)
    def emulate_int8(self, result, speedup=2.5):
        return EmuResult(result.name+"_int8", result.mac_ops, int(result.memory_bytes*0.25),
                         result.compute_units/speedup, result.memory_units/speedup,
                         result.effective_units/speedup, result.is_fused, result.fusion_memory_saved)

class GPUEmulator:
    def __init__(self, cuda_cores=128, mem_bandwidth_ratio=6.1, parallel_efficiency=None):
        self.cuda_cores = cuda_cores
        self.mem_bandwidth_ratio = mem_bandwidth_ratio
        self.parallel_efficiency = parallel_efficiency or {
            LayerType.CONV: 0.65, LayerType.DEPTHWISE_CONV: 0.20, LayerType.POINTWISE_CONV: 0.55,
            LayerType.BATCHNORM: 0.15, LayerType.RELU: 0.10, LayerType.POOL: 0.12, LayerType.FC: 0.50}
    def emulate_layer(self, layer):
        eff = self.parallel_efficiency[layer.layer_type]
        compute_units = layer.mac_ops / (self.cuda_cores * eff)
        memory_units = layer.memory_bytes_accessed / max(self.mem_bandwidth_ratio*eff, 0.01)
        return EmuResult(layer.name, layer.mac_ops, layer.memory_bytes_accessed, compute_units, memory_units, max(compute_units, memory_units))
    def emulate_fused_block(self, layers, pattern_name=""):
        tmac=tmem=tcu=0
        for i, layer in enumerate(layers):
            r = self.emulate_layer(layer); tmac += layer.mac_ops; tcu += r.compute_units
            if i == 0: tmem += layer.input_tensor_size_bytes + layer.param_count*4
            elif i == len(layers)-1: tmem += layer.param_count*4 + layer.output_tensor_size_bytes
            else: tmem += layer.param_count*4
        avg_eff = np.mean([self.parallel_efficiency[l.layer_type] for l in layers])
        mu = tmem / max(self.mem_bandwidth_ratio*avg_eff, 0.01)
        return EmuResult(f"fused_{pattern_name}", tmac, tmem, tcu, mu, max(tcu, mu), True)

# ----- FPGA efficiency dictionaries -----
# Trial 2 hand-set values (for comparison):
HANDSET_EFF = {
    LayerType.CONV: 0.45, LayerType.DEPTHWISE_CONV: 0.85, LayerType.POINTWISE_CONV: 0.65,
    LayerType.BATCHNORM: 0.90, LayerType.RELU: 0.95, LayerType.POOL: 0.90, LayerType.FC: 0.30}

# Trial 3 HLS-synthesized values (eff/DSP). RELU/POOL/FC not synthesized -> analogy.
_h = {k: v['efficiency_per_dsp'] for k, v in hls.items()}
GROUNDED_EFF = {
    LayerType.CONV:           _h['CONV'],
    LayerType.DEPTHWISE_CONV: _h['DEPTHWISE_CONV'],
    LayerType.POINTWISE_CONV: _h['POINTWISE_CONV'],
    LayerType.BATCHNORM:      _h['BATCHNORM'],
    LayerType.RELU:           _h['BATCHNORM'],     # analogy: elementwise ~ BN
    LayerType.POOL:           0.90,                # analogy: comparator tree (not synthesized)
    LayerType.FC:             _h['POINTWISE_CONV'],# analogy: matmul ~ pointwise
}

gpu_emu = GPUEmulator()
fpga_handset  = FPGAEmulator(streaming_efficiency=HANDSET_EFF)
fpga_grounded = FPGAEmulator(streaming_efficiency=GROUNDED_EFF)

print(f"{'op':<18}{'hand-set':>10}{'grounded(HLS)':>16}")
for lt in LayerType:
    tag = "" if lt in (LayerType.CONV,LayerType.DEPTHWISE_CONV,LayerType.POINTWISE_CONV,LayerType.BATCHNORM) else "  (analogy)"
    print(f"{lt.value:<18}{HANDSET_EFF[lt]:>10.3f}{GROUNDED_EFF[lt]:>16.3f}{tag}")

In [ ]:
# Cell 5 (from trial2) - Partition optimizer (unit space)
@dataclass
class PartitionResult:
    split_idx: int; gpu_blocks: int; fpga_blocks: int
    gpu_stage_units: float; fpga_stage_units: float; transfer_units: float
    @property
    def pipeline_bottleneck(self):
        if self.fpga_blocks == 0: return self.gpu_stage_units
        return max(self.gpu_stage_units, self.fpga_stage_units + self.transfer_units)
    @property
    def throughput(self):
        b = self.pipeline_bottleneck; return 1.0/b if b > 0 else 0
    @property
    def pipeline_efficiency(self):
        if self.fpga_blocks == 0: return 1.0
        g = self.gpu_stage_units; f = self.fpga_stage_units + self.transfer_units
        return min(g, f)/max(g, f) if max(g, f) > 0 else 0

def block_gpu_cost(block, gpu):
    return gpu.emulate_fused_block(block.layers, block.pattern).effective_units if isinstance(block, FusedBlock) else gpu.emulate_layer(block).effective_units

def block_fpga_cost(block, fpga, int8=True):
    if isinstance(block, FusedBlock):
        r = fpga.emulate_fused_block(block.layers, block.pattern)
        if int8: r = fpga.emulate_int8(r)
        return r.effective_units
    return fpga.emulate_layer(block).effective_units

def block_out_bytes(block):
    return block.output_tensor_size_bytes

def transfer_units(tensor_bytes, transfer_type="pcie"):
    if transfer_type == "pcie":
        return 50.0 + tensor_bytes / TARGET_XFER_BW_GBPS
    return 5.0 + tensor_bytes / (TARGET_XFER_BW_GBPS * 6.4)

def evaluate_partition(fused, split_idx, gpu, fpga, use_int8=True, transfer_type="pcie"):
    n = len(fused)
    if split_idx < 0 or split_idx >= n-1:
        return PartitionResult(-1, n, 0, sum(block_gpu_cost(b, gpu) for b in fused), 0, 0)
    gpu_total = sum(block_gpu_cost(b, gpu) for b in fused[:split_idx+1])
    fpga_total = sum(block_fpga_cost(b, fpga, use_int8) for b in fused[split_idx+1:])
    bbytes = block_out_bytes(fused[split_idx])
    if use_int8: bbytes //= 4
    return PartitionResult(split_idx, split_idx+1, n-split_idx-1, gpu_total, fpga_total, transfer_units(bbytes, transfer_type))

def find_optimal(fused, gpu, fpga, use_int8=True, transfer_type="pcie"):
    best = evaluate_partition(fused, -1, gpu, fpga, use_int8, transfer_type)
    allr = [best]
    for s in range(len(fused)-1):
        r = evaluate_partition(fused, s, gpu, fpga, use_int8, transfer_type)
        allr.append(r)
        if r.throughput > best.throughput: best = r
    return best, allr

print("Partition optimizer ready.")

In [ ]:
# Cell 6 - Optimal partition: grounded (HLS) vs hand-set FPGA costs (INT8, PCIe)
ARCHS = [("ResNet-18", build_resnet18), ("MobileNetV2", build_mobilenetv2)]
results = {}

for arch, builder in ARCHS:
    fused = apply_fusion(builder())
    gpu_only = sum(block_gpu_cost(b, gpu_emu) for b in fused)
    row = {"gpu_only": gpu_only}
    for tag, fpga in [("handset", fpga_handset), ("grounded", fpga_grounded)]:
        best, allr = find_optimal(fused, gpu_emu, fpga, use_int8=True, transfer_type="pcie")
        fpga_only = sum(block_fpga_cost(b, fpga, True) for b in fused)
        row[tag] = {
            "best": best, "all": allr,
            "gain_vs_gpu": (best.throughput / (1.0/gpu_only) - 1) * 100,
            "fpga_gpu_ratio": fpga_only / gpu_only,
            "split": best.split_idx, "gpu_blocks": best.gpu_blocks, "fpga_blocks": best.fpga_blocks,
            "eff": best.pipeline_efficiency * 100,
        }
    results[arch] = row

    print(f"\n{'='*72}\n{arch}  ({len(fused)} fused blocks)\n{'='*72}")
    print(f"  {'model':<12}{'FPGA/GPU':>10}{'opt split':>11}{'GPU|FPGA blk':>14}{'pipe eff':>10}{'gain vs GPU':>13}")
    for tag in ("handset", "grounded"):
        r = row[tag]
        print(f"  {tag:<12}{r['fpga_gpu_ratio']:>9.2f}x{r['split']:>11}"
              f"{str(r['gpu_blocks'])+'|'+str(r['fpga_blocks']):>14}{r['eff']:>9.1f}%{r['gain_vs_gpu']:>+12.1f}%")

In [ ]:
# Cell 7 - Three-way comparison (Analytical / Emulated hand-set / Grounded) + figure
# Trial 1 analytical numbers (from the paper / trial1.ipynb)
TRIAL1 = {"ResNet-18": {"gain": 0.8, "ratio": 36.9}, "MobileNetV2": {"gain": 11.8, "ratio": 9.4}}

print(f"{'='*86}")
print(f"THREE-WAY: throughput gain vs GPU-only  and  FPGA/GPU cost ratio")
print(f"{'='*86}")
print(f"  {'':<14}{'ResNet-18':>28}{'MobileNetV2':>28}")
print(f"  {'phase':<14}{'gain':>13}{'FPGA/GPU':>15}{'gain':>13}{'FPGA/GPU':>15}")
print(f"  {'-'*82}")
def _fmt(arch, which):
    if which == "analytical":
        return TRIAL1[arch]["gain"], TRIAL1[arch]["ratio"]
    return results[arch][which]["gain_vs_gpu"], results[arch][which]["fpga_gpu_ratio"]
for label, which in [("1 analytical", "analytical"), ("2 hand-set", "handset"), ("3 grounded", "grounded")]:
    g1, r1 = _fmt("ResNet-18", which); g2, r2 = _fmt("MobileNetV2", which)
    print(f"  {label:<14}{g1:>+12.1f}%{r1:>14.1f}x{g2:>+12.1f}%{r2:>14.1f}x")

# ---- figure ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle("Trial 3 - Hardware-Grounded FPGA Costs vs Analytical & Emulated", fontsize=14, fontweight='bold')

phases = ["Analytical\n(trial1)", "Emulated\nhand-set (trial2)", "Grounded HLS\n(trial3)"]
for ai, arch in enumerate(["ResNet-18", "MobileNetV2"]):
    gains = [TRIAL1[arch]["gain"], results[arch]["handset"]["gain_vs_gpu"], results[arch]["grounded"]["gain_vs_gpu"]]
    bars = axes[ai].bar(phases, gains, color=["#9E9E9E", "#FF9800", "#4CAF50"], alpha=0.9)
    axes[ai].axhline(0, color="k", lw=0.8)
    axes[ai].set_ylabel("Throughput gain vs GPU-only (%)")
    axes[ai].set_title(arch)
    axes[ai].grid(True, alpha=0.3, axis="y")
    for b, v in zip(bars, gains):
        axes[ai].text(b.get_x()+b.get_width()/2, v + (0.4 if v>=0 else -1.2), f"{v:+.1f}%", ha="center", fontsize=10, fontweight="bold")

# per-op efficiency: hand-set vs grounded
ax = axes[2]
ops = [LayerType.CONV, LayerType.DEPTHWISE_CONV, LayerType.POINTWISE_CONV, LayerType.BATCHNORM]
x = np.arange(len(ops)); w = 0.38
ax.bar(x - w/2, [HANDSET_EFF[o] for o in ops], w, label="hand-set (trial2)", color="#FF9800", alpha=0.9)
ax.bar(x + w/2, [GROUNDED_EFF[o] for o in ops], w, label="grounded HLS (trial3)", color="#4CAF50", alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels([o.value.replace("_conv","").replace("conv","conv") for o in ops], rotation=15)
ax.set_ylabel("FPGA efficiency (eff/DSP)")
ax.set_title("Per-op FPGA efficiency: assumed vs synthesized")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
outp = os.path.join(OUTPUT_DIR, "grounded_three_way_comparison.png")
plt.savefig(outp, dpi=150, bbox_inches="tight")
plt.show()
print("saved", outp)

## Findings & honesty

- **What changed:** only the FPGA per-op efficiency constants (hand-set →
  HLS-synthesized). GPU model, fusion, transfer, and partition search are
  identical to Trial 2, so any difference in the optimal split / gain is
  attributable to grounding the FPGA costs.
- **Direction of the correction:** synthesis shows the hand-set numbers were
  optimistic for the streaming ops on a per-DSP basis (esp. depthwise/pointwise
  as single-lane kernels) and that dense conv, while high-throughput when
  replicated, exceeds the 7020's DSP budget — consistent with the paper's thesis
  that dense conv is the FPGA's hard case.

**Before citing these numbers, close these gaps (tracked in `hls/README.md`):**
1. **Matched-parallelism kernels** — re-synthesize depthwise/pointwise/BN with
   comparable unrolling so `eff/DSP` is a fair cross-op comparison.
2. **Synthesize RELU/POOL/FC** instead of assigning them by analogy.
3. **Pragma sweep** on conv (`IC_UNROLL`) to report a latency-vs-DSP range, then
   propagate the range through a Monte-Carlo over the constants (phase-3 #6) to
   report gain distributions rather than point estimates.
4. **Calibration** — pair with the leave-one-out benchmark test in
   `calibration/` once more published rows are added.

This notebook is a **framework**: it re-reads `hls/reports/_fpga_constants.json`,
so as the kernels above are improved, re-running regenerates the comparison.
